In [1]:
!pip install --upgrade pip
!pip install qiskit qiskit-aer numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.2 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 117.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [qiskit-aer]3 [qiskit-aer]


In [2]:
# ================================
# COMPLETE QAOA KNAPSACK PROGRAM WITH EXPLANATIONS
# ================================

# ================================
# 1. IMPORT REQUIRED LIBRARIES
# ================================
"""
Think of these as importing different "tools" from a toolbox:

- NumPy: Like Excel for math - handles numbers, lists, random data
- QuantumCircuit (Qiskit): Blueprint paper for building quantum circuits
- Aer Simulator: Virtual quantum computer that runs on your regular laptop  
- SciPy Optimizer: Smart math algorithm that finds the best settings
"""

# NumPy is used to generate and handle numerical data (values, weights, random numbers)
import numpy as np

# QuantumCircuit is the basic object used to build quantum programs (quantum gates + qubits)
from qiskit import QuantumCircuit

# Aer provides a high-performance simulator that mimics a real quantum computer
from qiskit_aer import Aer

# SciPy optimizer is used for classical optimization of quantum parameters
from scipy.optimize import minimize

# ================================
# 2. FIX RANDOMNESS FOR REPRODUCIBILITY
# ================================
"""
Like setting a "random seed" in a video game - ensures we get the SAME random numbers
every time we run the program. Perfect for testing and comparing results.
"""
np.random.seed(42)

# ================================
# 3. DEFINE A LARGE KNAPSACK DATASET
# ================================
"""
Creates a REAL-WORLD-SCALE problem: 30 items to choose from.
Each item has a random value (profit/money) and weight.
Knapsack capacity = 200 units max weight.
"""
N = 30  # This simulates a LARGE dataset
values = np.random.randint(10, 100, N)  # Random profits: $10 to $99
weights = np.random.randint(5, 30, N)   # Random weights: 5 to 29 units
capacity = 200                          # Max knapsack weight

print("Generated", N, "items with values:", values[:5], "...")
print("Weights:", weights[:5], "... Capacity:", capacity)

# ================================
# 4. DEFINE THE KNAPSACK COST FUNCTION
# ================================
"""
TRANSLATES the knapsack problem into "quantum language".

Quantum computers don't understand "maximize profit".
They only understand "find LOWEST ENERGY state".

So we convert:
- High profit = LOW energy (good!)
- Over-weight = HIGH penalty energy (bad!!)

The computer will naturally find the lowest-energy (best profit) solution.
"""
def knapsack_cost(bitstring, values, weights, capacity):
    """
    bitstring = [0,1,1,0,1,...] where 1=select item, 0=don't select
    Returns "energy" score: LOWER = BETTER solution
    """
    # Total profit from selected items
    value = sum(values[i] * bitstring[i] for i in range(len(bitstring)))
    
    # Total weight from selected items  
    weight = sum(weights[i] * bitstring[i] for i in range(len(bitstring)))
    
    # Penalty if over capacity (squared = very harsh!)
    penalty = max(0, weight - capacity)
    
    # Energy = -profit + huge_penalty (quantum minimizes this)
    return -value + 10 * penalty**2

# ================================
# 5. BUILD THE QAOA QUANTUM CIRCUIT
# ================================
"""
The actual QUANTUM PROGRAM - like a recipe for quantum gates.

Two key parameters (gamma, beta) control how the algorithm works:
- gamma: How much we care about the problem (profit/weight)
- beta: How much we explore other possibilities

4 main steps:
1. Superposition: Try ALL combinations at once  
2. Cost layer: Score each combination by profit/weight
3. Mixer layer: Jump to better combinations
4. Measure: Pick the best one
"""
def qaoa_circuit(params, values):
    gamma, beta = params  # [gamma, beta] = tuning knobs
    n = len(values)       # Number of items = number of qubits
    
    qc = QuantumCircuit(n)
    
    # STEP 1: SUPERPOSITION - Try every possible combination simultaneously
    # Hadamard gate: 50% chance item selected, 50% not selected
    qc.h(range(n))  # Puts all qubits in magical "both 0 AND 1" state
    
    # STEP 2: COST HAMILTONIAN - Encode knapsack problem
    # RZ rotation: Items with high value get more "quantum phase preference"
    for i, v in enumerate(values):
        qc.rz(2 * gamma * v, i)  # Higher value = more rotation = lower energy
    
    # STEP 3: MIXER HAMILTONIAN - Explore the solution space
    # RX rotation: Allows jumping between different combinations
    qc.rx(2 * beta, range(n))  # "Mixes up" the superposition
    
    # STEP 4: MEASUREMENT - Collapse to classical answer
    qc.measure_all()  # Converts quantum weirdness → normal 0s and 1s
    
    return qc

# ================================
# 6. SET UP QUANTUM SIMULATOR
# ================================
"""
Instead of expensive real quantum hardware, we use a simulator.
It's like flight simulator training - feels real but runs on your laptop!
"""
backend = Aer.get_backend("aer_simulator")  # High-performance quantum simulator

# ================================
# 7. EXPECTATION VALUE FUNCTION
# ================================
"""
Tells us "How good are these quantum parameters?"

1. Build circuit with current parameters
2. Run 1024 times (shots) to get statistics
3. Average energy across all measured solutions
4. LOWER average = BETTER parameters
"""
def expectation(params, values, weights, capacity):
    qc = qaoa_circuit(params, values)
    
    # Run quantum circuit 1024 times
    job = backend.run(qc, shots=1024)
    counts = job.result().get_counts()  # {bitstring: frequency}
    
    total_energy = 0
    for bitstring, frequency in counts.items():
        # Convert "11001" → [1,1,0,0,1]
        bits = list(map(int, bitstring[::-1]))
        energy = knapsack_cost(bits, values, weights, capacity)
        total_energy += frequency * energy
    
    return total_energy / 1024  # Average energy

# ================================
# 8. BLOCK-WISE QAOA FOR SCALING
# ================================
"""
TRICK FOR BIG PROBLEMS: Current quantum computers can't handle 30 qubits.
Solution: Split into 8-qubit blocks, solve each separately, combine smartly.

Like solving a giant jigsaw puzzle by doing small sections first.
"""
def solve_knapsack_block_qaoa(values, weights, capacity, block_size=8):
    solution = []           # Final list of 0s and 1s
    remaining_capacity = capacity  # Updates as we fill knapsack
    
    # Process blocks of 8 items each
    for start in range(0, len(values), block_size):
        end = start + block_size
        v_block = values[start:end]    # Values for this block
        w_block = weights[start:end]   # Weights for this block
        
        print(f"\n{'='*50}")
        print(f"Solving block {start//8 + 1}: items {start}-{end-1}")
        print(f"Remaining capacity: {remaining_capacity}")
        
        # Classical optimizer finds BEST quantum parameters
        result = minimize(
            expectation,           # Function to minimize (energy)
            x0=[0.5, 0.5],         # Starting guess for [gamma, beta]
            args=(v_block, w_block, remaining_capacity),
            method="COBYLA"        # Smart optimization algorithm
        )
        
        print(f"Optimal parameters: gamma={result.x[0]:.3f}, beta={result.x[1]:.3f}")
        
        # Build final circuit with optimal parameters
        qc = qaoa_circuit(result.x, v_block)
        
        # SHOW THE ACTUAL QUANTUM CIRCUIT (proof it's quantum!)
        print("\nQuantum Circuit:")
        print(qc.draw(output="text"))
        
        # Run optimized circuit many times
        counts = backend.run(qc, shots=2048).result().get_counts()
        
        # Pick BEST solution from this block (lowest energy)
        best_bitstring = min(
            counts,
            key=lambda x: knapsack_cost(list(map(int, x[::-1])), v_block, w_block, remaining_capacity)
        )
        
        block_bits = list(map(int, best_bitstring[::-1]))
        print(f"Best bitstring: {best_bitstring} → {block_bits}")
        
        # Respect remaining knapsack capacity
        for i, (bit, weight_item) in enumerate(zip(block_bits, w_block)):
            if bit == 1 and weight_item <= remaining_capacity:
                solution.append(1)
                remaining_capacity -= weight_item
                print(f"  Item {start+i}: SELECTED (weight {weight_item})")
            else:
                solution.append(0)
                print(f"  Item {start+i}: SKIPPED")
    
    return solution

# ================================
# 9. RUN COMPLETE SOLUTION
# ================================
"""
Execute the full quantum-inspired algorithm and show results!
"""
print("\n" + "="*70)
print("🚀 LAUNCHING QUANTUM KNAPSACK SOLVER")
print("="*70)

solution = solve_knapsack_block_qaoa(values, weights, capacity)

# Calculate final results
total_value = sum(values[i] for i in range(len(solution)) if solution[i])
total_weight = sum(weights[i] for i in range(len(solution)) if solution[i])

print("\n" + "="*70)
print("🎉 FINAL SOLUTION FOUND!")
print("="*70)
print(f"Selected items: {solution}")
print(f"Total value: ${total_value:,.0f}")
print(f"Total weight: {total_weight}/{capacity}")
print(f"✅ FEASIBLE: {total_weight <= capacity}")
print("\n💡 This solves a 30-item knapsack using REAL quantum circuits!")


Generated 30 items with values: [61 24 81 70 30] ...
Weights: [20 19 19 23 16] ... Capacity: 200

🚀 LAUNCHING QUANTUM KNAPSACK SOLVER

Solving block 1: items 0-7
Remaining capacity: 200
Optimal parameters: gamma=1.716, beta=-0.475

Quantum Circuit:
        ┌───┐┌───────────┐ ┌─────────────┐ ░ ┌─┐                     
   q_0: ┤ H ├┤ Rz(209.3) ├─┤ Rx(-0.9495) ├─░─┤M├─────────────────────
        ├───┤├───────────┴┐├─────────────┤ ░ └╥┘┌─┐                  
   q_1: ┤ H ├┤ Rz(82.346) ├┤ Rx(-0.9495) ├─░──╫─┤M├──────────────────
        ├───┤├────────────┤├─────────────┤ ░  ║ └╥┘┌─┐               
   q_2: ┤ H ├┤ Rz(277.92) ├┤ Rx(-0.9495) ├─░──╫──╫─┤M├───────────────
        ├───┤├────────────┤├─────────────┤ ░  ║  ║ └╥┘┌─┐            
   q_3: ┤ H ├┤ Rz(240.17) ├┤ Rx(-0.9495) ├─░──╫──╫──╫─┤M├────────────
        ├───┤├────────────┤├─────────────┤ ░  ║  ║  ║ └╥┘┌─┐         
   q_4: ┤ H ├┤ Rz(102.93) ├┤ Rx(-0.9495) ├─░──╫──╫──╫──╫─┤M├─────────
        ├───┤├────────────┤├─────────────┤ ░  ║  ║ 